# Lab 11 · Keep the cases that match

**Today:** when you walk out, chapter 4's filtering computation, simulate, mask and count, is something you can write from a blank cell.

**Before you start:** lab 10's masks. Today they do the work on a simulated crowd.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down, on paper, out
loud, or in a comment. A prediction you can compare against the output is what
tells you which parts of the code you can already read.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. Every task has a hint in the **Hints**
block at the end of the notebook, for when you want it. The check never grades and never
breaks anything. A ⬜ means not attempted yet, a ❌ means not yet and comes with
a hint, and a ✅ means passing. Run the check cells rather than editing them.
Everything else in the notebook is yours to change.

**AI in this lab.** Until your prediction is written down, work at level 1, with
no AI. The prediction is how you find out what you can read unaided, and both
exams are level 1. Once you have run a cell, level 3 is encouraged: ask your
tutor to explain anything you missed.

Run every cell, and change things to see what happens. Nothing in this notebook
can be broken in a way that matters.

Ten thousand simulated mornings. Each morning is either a rush day, 30% of them, or a calm day, and the visitor count is a poisson draw at that kind of day's rate, 60 or 35, using `rng.poisson` from lab 7. Two arrays of the same length, positions aligned: `kind[i]` and `visitors[i]` describe the same morning, and the alignment is what lets a mask built on one array keep entries of the other. Read, then run.

In [ ]:
import numpy as np

rng = np.random.default_rng(101)
kind_list = []
visitor_list = []
for _ in range(10000):
    if rng.random() < 0.3:
        kind_list.append("rush")
        visitor_list.append(rng.poisson(60))
    else:
        kind_list.append("calm")
        visitor_list.append(rng.poisson(35))
kind = np.array(kind_list)
visitors = np.array(visitor_list)
print("rush share:", (kind == "rush").mean())

About 0.3, as built. The two aligned arrays are the labeled corpus of chapter 4, story removed.

**Test your understanding.** Create two variables from the crowd, using masks and no loops. `rush_mean` stores one number, the average visitor count on rush mornings, rounded to 1 decimal. `n_busy_calm` stores one whole number, how many *calm* mornings still had more than 50 visitors. This is question 1. Its hint is at the end of the notebook.

In [ ]:
# your turn: two variables named rush_mean and n_busy_calm (use the kind and visitors arrays above)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("rush_mean", expect=60.0,
      hint="mask from kind, filter visitors, .mean(), round 1")
check("n_busy_calm", expect=39,
      hint="two masks combined with & — calm AND above 50")

## 2 · The filtering answer

This morning saw **exactly 52 visitors**, and the door counter does not record the kind of day. Chapter 4's filtering computation, construct by construct: mask the crowd to mornings that look like this one, then ask what fraction of *those* were rush days. **Predict roughly, then run: with 52 sitting between the two rates, will the answer be near 0, near 1, or genuinely in between?**

In [ ]:
looks_like_today = (visitors == 52)
print("matching mornings:", looks_like_today.sum())
share_rush = (kind[looks_like_today] == "rush").mean()
print("share of them that were rush days:", round(share_rush, 2))

Genuinely in between: 52 is reachable by a busy calm day or a slow rush day, and the crowd's split among the matching mornings is the answer. That share is a probability, computed by counting. Chapter 4 calls it the **posterior**: a probability worked out after looking at the data, here the count of 52.

**Test your understanding.** Write a function named `share_rush_given` that takes three parameters: `count`, a whole number of visitors; `kind`, the array of morning kinds; and `visitors`, the array of visitor counts. It should return one number: the fraction of the simulated mornings with exactly `count` visitors that were rush days, rounded to 2 decimals. Then create two variables by calling it on the arrays above: `share_at_45`, the share at 45 visitors, and `share_at_50`, the share at 50. Predict the order before running the check: 45 sits nearer the calm rate, 50 nearer the middle of the two rates. This is question 2. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named share_rush_given(count, kind, visitors), then two variables named share_at_45 and share_at_50

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("share_at_45", expect=0.16,
      hint="mask on visitors == 45, then the rush share among the kept, rounded to 2")
check("share_at_50", expect=0.71,
      hint="the crossover region — most mornings at 50 are rush, but far from all")

## If you finish early

- Ask section 2 about a count of 47 or 48, between the two rates, and watch the answer pass through 0.5.
- Chapter 4's practice problem 4.1 asks the same question of the chapter's community with the mixing proportion changed.

## If you are stuck

Wave someone over. This hour exists so that a stuck step costs you a minute
rather than an evening. Known snags:

- **`rush_mean` is close but not exact.** The mask must come from `kind` and the mean from `visitors[is_rush]`; rounding to 1 decimal happens last.
- **`share_rush_given` errors on empty matches.** At counts nobody simulated, the filtered array is empty and `.mean()` warns. For today's checks the counts always match some mornings; the empty case is chapter 4's pseudocount discussion, worth rereading.
- **Everything passes but slowly.** A mask does 10,000 comparisons per line, which is normal and fast. The loop that would have done the same work is the one you did not have to write.

## Hints

**Question 1 · `rush_mean`, `n_busy_calm`.** Build a mask on `kind` first: `is_rush = (kind == "rush")`. Then `visitors[is_rush].mean()`, rounded to 1 decimal, is `rush_mean`, the way lab 10's section 2 filtered one array with a mask built from another. For `n_busy_calm`, combine two masks with `&`, as in lab 10's section 3: `(kind == "calm") & (visitors > 50)`, each comparison in its own parentheses, then `.sum()`.

**Question 2 · `share_rush_given`.** Section 2's two lines are the function body with 52 replaced by `count`. Inside the function, build `looks_like = (visitors == count)`, then return `round((kind[looks_like] == "rush").mean(), 2)`. Then two plain assignment lines call it: `share_at_45 = share_rush_given(45, kind, visitors)`, and the same at 50.